# Build and validate a graph

Learn to diagnose a declared dtype mismatch, confirm a corrected graph, and submit values for execution. You should already know basic Python and what NumPy arrays and PyTorch tensors are; no training expertise is needed. The tiny model only gives us a concrete payload to pass between steps.

## Before you begin

Run the cells in order in a fresh kernel. Keep the complete repository and bundled tutorial data together. The setup below locates your checkout; in Colab it accepts a complete source ZIP. The notebook imports the adjacent Python implementation, so operators and resources have one canonical source.


In [ ]:
from pathlib import Path
import importlib.util
import io
import subprocess
import sys
import zipfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # Upload the complete Jayrun project ZIP, not just this notebook.
    if len(uploaded) != 1:
        raise ValueError("upload exactly one project ZIP")
    destination = Path("/content/jayrun_tutorial_source")
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        if sum(item.file_size for item in archive.infolist()) > 500_000_000:
            raise ValueError("archive exceeds this notebook's extraction limit")
        for item in archive.infolist():
            path = (destination / item.filename).resolve()
            if not path.is_relative_to(destination.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("unsafe archive path or symbolic link")
        archive.extractall(destination)
    candidates = [destination, *destination.iterdir()]
    ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run Jupyter from the extracted project, or upload the full project ZIP in Colab.")
sys.path.insert(0, str(ROOT))
print("Tutorial source:", ROOT)


## Prepare the environment

The next cell checks the packages this lesson uses. Locally, install missing tutorial packages before continuing. In Colab, it installs only missing non-PyTorch packages. For PyTorch lessons, provide a suitable existing PyTorch build; the cell does not replace it.


In [ ]:
required = {'packaging': 'packaging>=25', 'numpy': 'numpy', 'torch': 'torch'}
missing = [requirement for name, requirement in required.items() if importlib.util.find_spec(name) is None]
# Colab provides PyTorch. Never replace an existing CPU/CUDA installation here.
if "torch" in missing:
    raise RuntimeError("Install the PyTorch build appropriate for your hardware before running this lesson.")
if missing:
    if not IN_COLAB:
        raise RuntimeError(f"Install these optional tutorial dependencies first: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
from packaging.version import Version
import packaging
if Version(packaging.__version__) < Version("25"):
    raise RuntimeError("This source archive requires packaging >= 25; update it explicitly before proceeding.")
import torch
torch.set_num_threads(1)  # Keep the small CPU demonstration bounded.


## 1. Follow the two flows

The dataset flows through **PrepareDataset → TrainModel**. A model flows through **TrainModel → ScaleModel**. TrainModel waits for both inputs. An Artifact names a data position; its value belongs to a submission. An operator that outputs the same artifact replaces that value along its flow.

The builder declares both entry flows because the application supplies both the initial table and the initial model. Inspect it before executing anything.


In [ ]:
import asyncio
import inspect
import numpy as np
import torch
from jayrun import ArtifactContext, Engine
from jayrun.context import ContextState
from tutorials import build_graph as lesson

print(inspect.getsource(lesson.build_graph))


## 2. Diagnose a declaration without running it

PrepareDataset promises the dtype selected by the builder. TrainModel requires float32. Choose float64 and inspect the report: exactly one producer/consumer edge should be incompatible. Validation compares declared properties; it does not examine actual input arrays.


In [ ]:
bad, _, _ = lesson.build_graph(torch.float64)
validation = bad.validate()
print("Valid:", validation.valid)
print("Mismatched edges:", len(validation.mismatched_edges))
assert not validation.valid and len(validation.mismatched_edges) == 1

graph, dataset, model = lesson.build_graph(torch.float32)
assert graph.validate().valid
graph.confirm()
print("Corrected graph confirmed:", graph.confirmed)


## 3. Supply values and inspect the outcome

The table has 784 feature columns and one label column. ArtifactContext maps the two entry identities to a table and a classifier. Constructors declare fields; Jayrun binds artifact arguments to fields by name and output tuples by position. The Engine returns a ContextRun, whose final state must be checked before reading a result.

Jupyter owns an event loop, so the synchronous Engine block runs through asyncio.to_thread. This is orchestration code using the existing tutorial operators.


In [ ]:
def execute_corrected_graph():
    rng = np.random.default_rng(42)
    table = np.column_stack((rng.normal(size=(24, 784)), np.arange(24) % 10))
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(42)
        classifier = torch.nn.Linear(784, 10)
    inputs = ArtifactContext({dataset: table, model: classifier})
    with Engine() as engine:
        run = engine.submit(graph, inputs)
        run.wait(timeout=30)
        if run.state is not ContextState.FINISHED:
            raise RuntimeError(str(run.report.data.failure))
        return run.artifact(model).value

result = await asyncio.to_thread(execute_corrected_graph)
print("Result type:", type(result).__name__)
assert isinstance(result, lesson.TemperatureScaledModel)


## 4. Try another declaration

Predict what happens if the producer declares float16. Run the cell to check your prediction. Then change float16 to float32: the mismatch count should become zero. A fixed declaration still does not prove model quality or runtime payload correctness.


In [ ]:
alternative, _, _ = lesson.build_graph(torch.float16)
print("Alternative mismatches:", len(alternative.validate().mismatched_edges))


You have separated declaration, validation, confirmation, submission, and result inspection. The complete CLI demonstration also returns mismatches_detected=1, corrected_valid=True, and result_type="TemperatureScaledModel". Continue to the image lesson for conditional routes and reusable resources.
